# Exploratory Data Analysis

Generated from `notebooks/DELE_CA1_B.ipynb` by `scripts/split_notebook.py`.


---

# 3. Exploratory Data Analysis

In this section, we will be conducting Visualisations on our Cleaned Dataset to determine the best way forward for the Training of Models and also to understand where our Data might be weak in and how we can potentially compensate for it. This section will be separated into 2 different section, one for Classification and the other for Regression. This is because we have not yet decided whether we want to perfrom Classification or Regression on our Data.

---
## 3.1 Overview Visualisation

In this sub-section, we will be mainly focusing on the Data Visualisation that are relevant to the Dataset as a whole. Through this sub-section, we hope to better understand our data and what can we do to better improve the RNN Model's performance in general for both Regression and Classification.

---
### 3.1.1 Review Language Distribution

In this sub-section, we will be utilising a bar chart to visualise the Distribution of Review Languages within the dataset. The purpose of this visualisation has been indicated below:

- Identifies the linguistic composition of the dataset.
- Assists in determining whether language-specific preprocessing may be necessary.
- Provides clarity on how language diversity might affect both model learning and generalisability.
- Supports the decision to filter or group reviews by language for more consistent modelling.

With the purpose indicated, we will proceed to conduct the visualisation in the Code Cell below.

In [ ]:
# ==================== Language Distribution ==================== #
fig_lang = px.histogram(
    df,
    x='Language',
    color='Language',
    title='Review Language Distribution',
    color_discrete_sequence=px.colors.qualitative.Bold
)

fig_lang.update_layout(
    xaxis_title='Language',
    yaxis_title='Number of Reviews',
    template='plotly_white',
    title_font=dict(size=20),
    showlegend=False
)

fig_lang.show()

With reference to the output of the Visualisation above, we are able to make to following observations.

- The majority of reviews are in English followed by a substantial portion in **Malay**.
- Only 1 review are in Chinese, which may be too limited for effective modelling. Thus it will be dropped.
- This suggests the dataset is primarily bilingual (English and Malay), allowing for effective sentiment analysis using either or both languages.
- Any multilingual modelling approach must either translate non-English/Malay reviews or exclude them to maintain data consistency.
- There is a class imbalance between Malay and English. However as the text will be tokenised, the class imbalance is not a major issue.

With the observations and implications indicated, we will proceed to the next

---
#### 3.1.1.1 Removing Chinese Review

As visualised previously, the Language with the least Review is Chinese which means that it will not only be an issue when training the RNN Model, but due to the lack of data, it is virtually useless as the RNN Model will be unable to pick out any useful Data from it. Therefore, the best course of action to undertake in this case is to drop the Chinese Language Review. This operation will be conducted in the Code Cell below.

In [ ]:
# ==================== Dropping Chinese Reviews ==================== #
df = df[df['Language'].str.lower() != 'chinese'].reset_index(drop=True)

# ==================== Displaying DataFrame ==================== #
df.head(5).style.background_gradient(cmap="Blues")

,Review,Language,word_count,Sentiment_Binary,Sentiment_Label,Score
0,A big surprise in the middle of the film! Thrilling action!,English,11,0,Positive,0.880000
1,A big surprise in the plot! Thrilling action throughout.,English,9,0,Positive,0.900000
2,A cinematic experience that is unforgettable. I'm impressed!,English,8,0,Positive,0.900000
3,"A cinematic marvel! The visuals are breathtaking, and the narrative is spellbinding.",English,12,0,Positive,0.920000
4,A complex yet engaging plot. A major surprise awaits at the end!,English,12,0,Positive,0.940000


With reference to the Output above, we are able to see that the dropping have been conducted successfully for the Data Row with the Chinese Language.

---
### 3.1.2 Distribution of Review Length

In this sub-section, we will be utilising a histogram to display the Distribution of Review Word Counts across pre-defined bins. The purpose of this visualisation has been indicated below:

- Provides an overview of how long or short the reviews are within the dataset.
- Informs the choice of a suitable padding length during tokenisation, especially for RNNs.
- Helps detect outlier bins that might require review trimming or special handling.
- Ensures that model input sequences are efficiently processed while maintaining semantic richness.

With the purpose indicated, we will proceed to conduct the visualisation in the Code Cell below.

In [ ]:
# ==================== Calculate Word Count ==================== #
df['word_count'] = df['Review'].apply(lambda x: len(str(x).split()))

# ==================== Word Count Bins (Clean & Sorted X-Axis) ==================== #
bin_edges = [0, 10, 20, 30, 50, 100, 200, 1000]
bin_labels = ['0–10', '11–20', '21–30', '31–50', '51–100', '101–200', '201+']

df['word_count_bin'] = pd.cut(df['word_count'], bins=bin_edges, labels=bin_labels, include_lowest=True)

fig_bins = px.histogram(
    df,
    x='word_count_bin',
    category_orders={'word_count_bin': bin_labels},
    title='Distribution of Review Length Bins',
    color_discrete_sequence=['#9467bd']
)

fig_bins.update_layout(
    xaxis_title='Review Word Count Range',
    yaxis_title='Number of Reviews',
    template='plotly_white',
    title_font=dict(size=20)
)

df.drop("word_count_bin", axis=1, inplace=True)

fig_bins.show()

With reference to the output of the Visualisation above, we are able to make the following observations.

- Most reviews fall within the 0–10 and 11–20 word ranges, confirming that reviews are generally short.
- A noticeable drop is observed in the 21–30 word bin, indicating that only a small fraction of users provide more detailed reviews.
- Moderate presence is seen in the 31–50 range, showing a small group of users who elaborate more extensively.
- Very few reviews extend beyond 50 words, suggesting that long reviews are rare and might not heavily influence model training.
- These insights support a padding length around 50 as sufficient for covering the majority of review lengths without excessive truncation or computational cost.

With the observations and implications indicated, we will proceed to conduct the next Section of Visualisation.

---
## 3.2 Regression Data Visualisation

In this sub-section, we will be mainly focusing on the Data Visualisation that are relevant for Regression. Through this sub-section, we hope to better understand our data and what can we do to better improve the RNN Model's performance in general for Regression.

---
### 3.2.1 Distribution of Sentiment Scores

In this sub-section, we will be utilising a histogram with a marginal box plot to show the distribution of Sentiment Scores across the dataset. The purpose of this visualisation has been indicated below:

- Helps determine whether the dataset is suitable for a Regression Modelling Task.
- Reveals if the Score Values are Skewed or Uniformly Distributed, which affects how well the model can generalise.
- Highlights the concentration of reviews around certain score ranges, informing class balance concerns if scores are later converted into classification labels.
- Provides insight into whether any transformation or augmentation is needed before training.
- The marginal box plot allows us to detect statistical outliers and better understand the spread of scores.

With the purpose indicated, we will proceed to conduct the visualisation in the Code Cell below.

In [ ]:
# ==================== Score Distribution (Regression) ==================== #
fig_score = px.histogram(
    df,
    x='Score',
    nbins=30,
    marginal='box',
    color_discrete_sequence=['#1f77b4'],
    title='Distribution of Sentiment Scores (Regression Target)'
)

fig_score.update_layout(
    xaxis_title='Sentiment Score (0 to 1)',
    yaxis_title='Number of Reviews',
    template='plotly_white',
    title_font=dict(size=20),
)
fig_score.show()

With reference to the output above, we are able to make the following observations.

- The majority of sentiment scores are heavily concentrated between 0.05 and 0.15, indicating a strong skew towards Negative Sentiment.
- Very few reviews have scores above 0.6, which corresponds to the Positive Class Threshold in the classification task.
- The skewed distribution confirms the presence of Label Imbalance, which can affect both regression and classification performance.
- The box plot further reveals that the Median Score is Extremely Low (~0.1), and the upper whisker extends towards high scores, though with few data points.
- These insights justify the need for **class weighting and data augmentation** to mitigate model bias during training.

With the observations and implications indicated, we will proceed to conduct the next Regression Visualisation.

---
### 3.2.2 Review Length vs Score

In this sub-section, we will be utilising a scatter plot to visualise the relationship between Review Length (Word Count) and the corresponding Score. The purpose of this visualisation has been indicated below:

- Helps determine whether review length has any correlation with sentiment scores.
- Assists in understanding if longer reviews tend to be more positive or more detailed.
- Supports the decision-making for input truncation and padding length especially in the RNN architecture.
- Provides visual evidence of potential biases in short reviews.

With the purpose indicated, we will proceed to conduct the visualisation in the Code Cell below.

In [ ]:
# ==================== Review Length vs Sentiment Score ==================== #
fig_scatter = px.scatter(
    df,
    x='word_count',
    y='Score',
    title='Review Length vs Sentiment Score',
    color='Score',
    color_continuous_scale='Viridis',
    opacity=0.7
)

fig_scatter.update_layout(
    xaxis_title='Word Count',
    yaxis_title='Score',
    template='plotly_white',
    title_font=dict(size=20),
)
fig_scatter.show()

With reference to the Output above, we are able to make the following observations.

- The majority of reviews are less than 15 words, with a visible cluster of very short reviews (~10 words) dominating the dataset.
- There is no strong linear correlation between review length and sentiment score, but a pattern emerges:
  - Many short reviews (under 10 words) tend to have very low sentiment scores, suggesting brief complaints or harsh feedback.
  - Higher sentiment scores (above 0.6) appear more frequently among reviews with moderate to long lengths, indicating that positive sentiment is often expressed more elaborately.
- The pattern supports the earlier decision to remove ultra-short reviews (< 4 words) during data cleaning.
- This scatter plot reaffirms that padding length must be chosen carefully, and that review length does not guarantee sentiment strength, but may influence expression style.

With the observations indicated, we will proceed to the next Section.

---

## 3.3 Classification Data Visualisation

In this sub-section, we will be mainly focusing on the Data Visualisation that are relevant for Classification. Through this sub-section, we hope to better understand our data and what can we do to better improve the RNN Model's performance in general for Classification.

---
### 3.3.1 Visualising Class Imbalance

In this sub-section, we will be visualising the quantity count for each of the sentiment class so as to identify any potential Class Imbalance. Class Imbalance could cause skewed accuracy as the model is able to better learn from the sentiment class with higher quantity of data whereas sentiment classes with lower number of data will cause a lower accuracy. This can cause a high accuracy model as the model may be extremely capable of identifying a specific class while failing to identify other classes. The various actions that can be taken to address Class Imbalance are listed below:

**Methods To Address Class Imbalance**
- Establishing Class Weights
- Oversampling / Undersampling

With the various Methods to Address Class Imbalance, we shall proceed to state the appropriate action plan for various outcomes of the visualisation. The action plan for each respective results are listed below:

**Class Imbalance Identified**
- List down the observations from the Barplot.
- Proceed to address Class Imbalance using one of the listed method.
- Justify the chosen method's usage over other method.

**No Class Imbalance Identified**
- List down the observations from the Barplot.
- Proceed to next visualisation.

With the action plan for the potential results listed, we will proceed to conduct the visualisation in the Code Cell below.

In [ ]:
# ==================== Ternary Sentiment Class Distribution ==================== #
fig_class = px.histogram(
    df,
    x='Sentiment_Label',
    color='Sentiment_Label',
    title='Binary Sentiment Class Distribution',
    color_discrete_map={
        'Positive': '#2ca02c',
        'Negative': '#d62728'
    }
)

fig_class.update_layout(
    xaxis_title='Sentiment Class',
    yaxis_title='Number of Reviews',
    template='plotly_white',
    showlegend=False,
    title_font=dict(size=15),
)

fig_class.show()

With reference to the output of the output of the barplot above, we are able to see that there is a Significant **Class Inbalance** within our Dataset. This could cause the RNN Model to be better in predicting Positive Reviews as compared to Positive or Neutral Reviews due to the likihood of a higher amount of Training Data using Positive Reviews. Therefore, to address the Class Imbalance, we will have to identify a method that is suitable in this case.

---
### 3.3.2 Score Distribution by Sentiment

In this sub-section, we will be utilising the violin plot to visualises the Distribution of Scores across the three Sentiment Classes: Negative, Neutral, and Positive. The potential obsevations that we can gather from this Visualisation are indicated below.

- Validating the label thresholds used to generate the ternary classification target from the continuous Score
- Understanding whether the classes are well-separated in terms of their score distribution
- Confirming that the model has meaningful signal to learn from.

By including individual data points along with a smoothed density curve and boxplot, this visual provides both high-level and detailed insights into how sentiment scores are distributed within each class. With the purpose and potential observations indicated, we will proceed to conduct the Visuaisation in the Code Cell below.

In [ ]:
# ==================== Score Distribution by Sentiment (Violin Plot) ==================== #
fig_violin = px.violin(
    df,
    x='Sentiment_Label',
    y='Score',
    color='Sentiment_Label',
    box=True,
    points='all',
    title='Score Spread by Sentiment Class',
    color_discrete_map={
        'Negative': '#d62728',
        'Neutral': '#1f77b4',
        'Positive': '#2ca02c'
    }
)

fig_violin.update_layout(
    xaxis_title='Sentiment Class',
    yaxis_title='Score',
    template='plotly_white',
    title_font=dict(size=20),
)
fig_violin.show()

With reference to the output of the Violin Plot above, we are able to make the following observations.

- **Negative** reviews are tightly clustered around scores between **0.1 and 0.35**, showing strong separation from the other classes.
- **Positive** reviews are consistently within the **0.7 to 0.8+ range**, with minimal overlap into Neutral territory.
- There is **clear separation** between the classes, which justifies the chosen thresholds (0.4 and 0.7) for converting regression scores into ternary classification labels.

With the observations indicated, we will proceed to the next Visualisation for Classification.

---
### 3.3.3 Review Length by Sentiment Class

In this sub-section, we will be utilisng box plot to shows the Distribution of Review Lengths (word count) across the Three Sentiment Classes. The purpose of the Visualisation have been indicated below.

- Helps determine if there is a relationship between Sentiment and Review Length.
- Informs whether different sentiment classes requires Different Padding Lengths in the tokenisation phase.
- Helps detect outliers in Review Length.

Understanding the variation in review lengths ensures the RNN Model is appropriately designed to handle different input sizes and may reveal behavioural patterns in how users express sentiment. With the purpose indicated, we will proceed to conduct the Visualisation in the Code Cell below.

In [ ]:
# ==================== Review Length by Sentiment Class (Box Plot) ==================== #
fig_box = px.box(
    df,
    x='Sentiment_Label',
    y='word_count',
    color='Sentiment_Label',
    title='Review Length by Sentiment Class',
    color_discrete_map={
        'Negative': '#d62728',
        'Neutral': '#1f77b4',
        'Positive': '#2ca02c'
    }
)

fig_box.update_layout(
    xaxis_title='Sentiment Class',
    yaxis_title='Review Word Count',
    template='plotly_white',
    title_font=dict(size=20),
)
fig_box.show()

With reference to the output above, we are able to make the following Observations.

- Positive Reviews show a wide range of lengths, with many short reviews, likely due to brief complaints.
- Negative Reviews tend to be longer on average, with a higher median word count, possibly reflecting more detailed expressions of appreciation.
- The broader spread across classes supports the use of a Uniform Padding Length for Tokenised Inputs.

With the observations and implications indicated, we will proceed to Visualise Regression-related EDAs.